In [41]:
import os
import pandas as pd
import duckdb as db
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import joblib
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_auc_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import numpy as np
from sklearn.impute import SimpleImputer

In [11]:
load_dotenv()
db_path = os.getenv("DATABASE_PATH")
conn = db.connect(db_path)
df = conn.sql("""Select * FROM processed_news
""").df()

In [13]:
df['full_text'] = df['title'] + " " + df['text']

In [29]:
X = df.drop(['title', 'text', 'subject', 'is_fake', 'date'], axis=1)
y = df['is_fake']
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42, test_size=0.2)

In [30]:
custom_stop_words = list(ENGLISH_STOP_WORDS) + ['reuters', 'said', 'via', 'twitter', 'video', 'watch', 'breaking', 'shared']
num_cols = X.select_dtypes(include = np.number).columns.tolist()
text_cols = X.select_dtypes(exclude = np.number).columns.tolist()

In [42]:
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
     ('scaler', MinMaxScaler())])
preprocessor = ColumnTransformer([
    ('num', num_transformer, num_cols),
    ('text', TfidfVectorizer(stop_words=custom_stop_words), 'full_text')])

In [43]:
models_to_train = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "XGBoost": xgb.XGBClassifier(n_jobs=-1, random_state=42)
}
results = []
best_accuracy = 0
best_model = ""
best_pipeline = None

for name, model in models_to_train.items():
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    probs= pipe.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    roc_auc = roc_auc_score(y_test, probs)
    print(f'{name} accuracy {acc}, f1 {f1} roc_auc_score {roc_auc}')
    results.append({
        "Model": name,
        "Accuracy": acc,
        "F1-Score": f1,
        "Roc_Auc_Score": roc_auc
    })

    if acc > best_accuracy:
        best_accuracy = acc
        best_model = name
        best_pipeline = pipe
        
results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False)

conn.close()

Logistic Regression accuracy 0.9797953964194374, f1 0.977645727221279 roc_auc_score 0.997674712764836
Naive Bayes accuracy 0.9324808184143223, f1 0.9243553008595988 roc_auc_score 0.981969998418889
XGBoost accuracy 0.9884910485933504, f1 0.987363100252738 roc_auc_score 0.9993711789817646


In [49]:
load_dotenv()
joblib_dir = os.getenv("JOBLIBPATH")

file_path = os.path.join(joblib_dir, f"{best_model}_pipeline_num.joblib")

joblib.dump(best_pipeline, file_path)

['D:\\IT\\Python\\fake-news-detection\\models\\XGBoost_pipeline_num.joblib']